# Day 11 · Senses: Sound — What You Hear While You Wait

**Question.** The systems that move the city also make its noise: expressways, the 'L', freight
trains, aircraft. Riders wait inside it. How loud is it where people wait for a train or a bus, and
where people live?

**Approach.** The Bureau of Transportation Statistics' National Transportation Noise Map (2022
release) models road, rail and aviation noise as a 24-hour average (LAeq,24h) on 30 m cells. Combine
the three modes, take the median within 90 m of every point (the model puts the listener on the
track or roadway itself, so a single cell is unreliable next to a source), and read it at every 'L'
platform, CTA bus stop and Census block. Check the model against the noise the Chicago Department
of Aviation measures around Midway.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, settings |
| 2. Pull | Noise rasters (clipped), 'L' stations, CTA GTFS stops, 2020 blocks, Midway monitors |
| 3. Validate | Grids align, value ranges, counts |
| 4. Surface | Combine the modes by energy; 90 m median; dominant mode |
| 5. Sample | Platforms → stations, bus stops, residents, monitors |
| 6. Analyze | Shares, lines, the finding |
| 7. QA view | Quick map and tables |
| 8. Export | `plate.json` |

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import subprocess
import zipfile
from datetime import date
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import shapely
from pyproj import Transformer
from rasterio.transform import rowcol
from scipy import ndimage
from shapely.geometry import Polygon

from atlas import Ledger, write_plate
from atlas import grid

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day11')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
DAY = 11
MODES = ['road', 'rail', 'aviation']
FLOOR = 40.0          # dB given to cells the model leaves empty (it maps nothing below 45 dB)
RADIUS_M = 90         # median window around every point: about half a block
CELL_M = 30
CLASSES = [45, 50, 55, 60, 65, 70, 75]   # dB steps for the residents' shares
LOUD = 65             # loud: FAA/HUD's line for significant noise (stated as a round reference)
EPA = 70              # EPA 1974: 24-hour average that protects hearing over a lifetime
LATTICE = 8           # dots per mile: one per city block (100 address numbers)

ROOT = Path('../..').resolve()
DAY_DIR = Path('.').resolve()
NOISE_DIR = ROOT / 'reference' / 'raw' / 'noise'
CLIP_DIR = NOISE_DIR / 'clip'
GTFS = sorted((ROOT / 'reference' / 'raw' / 'gtfs').glob('cta_gtfs_*.zip'))[-1]
DAY08 = ROOT / 'days' / 'day08-utopia' / 'output' / 'plate.json'
BLOCKS = ROOT / 'reference' / 'chicago-blocks-2020.csv'
CITY = ROOT / 'reference' / 'city-boundary.json'
SEQ = ROOT / 'reference' / 'cta-line-sequences.json'
RAW = DAY_DIR / 'data' / 'raw'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'

BTS_PAGE = 'https://www.bts.gov/geospatial/national-transportation-noise-map'
BTS_DOC = 'https://rosap.ntl.bts.gov/view/dot/90910'
BTS_ZIP = 'https://www.bts.gov/bts-net-storage/CONUS_{}_noise_2022.zip'
MDW_REPORT = ('https://www.flychicago.com/SiteCollectionDocuments/Community/AccessiblePDFs/'
              'Noise/Midway/ANMS/MDW_4Q2025_ANMS_Report.pdf')
EPA_URL = 'https://epa.gov/archive/epa/aboutepa/epa-identifies-noise-levels-affecting-health-and-welfare.html'
CTA_STUDY = 'https://stacks.cdc.gov/view/cdc/203765'  # free full text; doi 10.1080/15459624.2017.1285490
# Clip box: city + 1 mile in USA Contiguous Albers (ESRI:102039), metres
PROJWIN = (660645, 2145128, 701420, 2100088)

## 2. Pull

The BTS files are continental mosaics (FileGDB, one raster per state) that bts.gov serves only to a
browser. They were downloaded by hand into `reference/raw/noise/` (gitignored) and the Illinois
rasters cut to the city plus a mile with GDAL. The cell below re-cuts them if the clips are missing
and the geodatabases are unzipped; otherwise it reads the clips.

In [ ]:
def clip(mode: str) -> Path:
    out = CLIP_DIR / f'{mode}_chicago.tif'
    if out.exists():
        return out
    gdb = NOISE_DIR / f'CONUS_{mode}_noise_2022' / f'CONUS_{mode}_noise.gdb'
    if not gdb.exists():
        raise FileNotFoundError(f'Unzip {BTS_ZIP.format(mode)} into {NOISE_DIR} first (Illinois tables are enough)')
    CLIP_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['gdal_translate', '-q', '-projwin', *map(str, PROJWIN), '-co', 'COMPRESS=DEFLATE',
                    f'OpenFileGDB:{gdb}:IL_{mode}_noise', str(out)], check=True)
    return out


layers, meta = {}, {}
for m in MODES:
    with rasterio.open(clip(m)) as d:
        layers[m] = d.read(1, masked=True).filled(np.nan).astype('float64')
        meta[m] = (d.transform, d.crs, d.shape)
TRANSFORM, CRS, SHAPE = meta['road']
log.info('Rasters %s cells of %d m, %s', SHAPE, CELL_M, CRS)

In [ ]:
d8 = json.loads(DAY08.read_text())
stations = pd.DataFrame(d8['stations'])
stations = stations[stations['id'].str.isdigit() & stations['inCity']].copy()
stations['id'] = stations['id'].astype(int)
rail_track = d8['rail']

zf = zipfile.ZipFile(GTFS)
stops = pd.read_csv(zf.open('stops.txt'), dtype={'parent_station': 'Int64'})
platforms = stops[(stops['stop_id'] >= 30000) & (stops['stop_id'] < 40000)].copy()
bus = stops[stops['stop_id'] < 30000].copy()
GTFS_DATE = date.fromisoformat(GTFS.stem.split('_')[-1])

blocks = pd.read_csv(BLOCKS)
city_rings = json.loads(CITY.read_text())['rings']
city = Polygon()
for r in city_rings:  # rings drawn even-odd: enclaves such as Norridge are holes
    city = city.symmetric_difference(shapely.make_valid(Polygon(r)))

monitors = pd.read_csv(RAW / 'mdw_monitors_4q2025.csv', dtype={'monitor': str}).merge(
    pd.read_csv(RAW / 'mdw_monitors_geocoded.csv', dtype={'monitor': str}), on='monitor')
seq = json.loads(SEQ.read_text())['lines']
log.info('%d city stations, %d platforms, %d bus stops, %d blocks, %d monitors (GTFS %s)',
         len(stations), len(platforms), len(bus), len(blocks), len(monitors), GTFS_DATE)

## 3. Validate

In [ ]:
for m in MODES:
    assert meta[m][0] == TRANSFORM and meta[m][2] == SHAPE, f'{m}: grid differs from road'
    v = layers[m][np.isfinite(layers[m])]
    assert v.min() >= 45 - 1e-3, f'{m}: values below the 45 dB cut-off'
    log.info('%-8s cells with a value %6.1f%%  median %.1f  max %.1f', m, 100 * v.size / layers[m].size, np.median(v), v.max())
assert len(stations) == 124, len(stations)
NO_PLATFORM = sorted(set(stations['id']) - set(platforms['parent_station'].dropna().astype(int)))
log.info('Stations without a GTFS platform (read at the station point): %s',
         stations.loc[stations['id'].isin(NO_PLATFORM), 'short'].tolist())
assert blocks['pop'].sum() == 2_746_388
assert monitors['lat'].notna().all() and len(monitors) == 13

## 4. Surface

Modes combine by energy: L = 10·log10(Σ 10^(Lᵢ/10)). Cells the model leaves empty are below 45 dB;
they count as 40 dB so a median can still be taken. The median within 90 m (29 cells) replaces the
single cell: next to a source the model reads 80–95 dB in the one cell the track or roadway runs
through and about 60 dB one cell away, so a point's value would depend on which cell it lands in.

In [ ]:
yy, xx = np.mgrid[-3:4, -3:4]
FOOT = np.hypot(yy, xx) * CELL_M <= RADIUS_M
filled = {m: np.where(np.isfinite(layers[m]), layers[m], FLOOR) for m in MODES}
energy = sum(np.where(np.isfinite(layers[m]), 10 ** (layers[m] / 10), 0.0) for m in MODES)
combined = np.where(energy > 0, 10 * np.log10(np.maximum(energy, 1e-12)), FLOOR)

surface = ndimage.median_filter(combined, footprint=FOOT, mode='nearest')
by_mode = np.stack([ndimage.median_filter(filled[m], footprint=FOOT, mode='nearest') for m in MODES])
dominant = np.where(by_mode.max(axis=0) >= 45, by_mode.argmax(axis=0), -1)


def block_average(width_cells: int) -> np.ndarray:
    """The map's dots: the energy average of the surface over a square about a block (or half) wide."""
    e = ndimage.uniform_filter(10 ** (surface / 10), size=width_cells, mode='nearest')
    return 10 * np.log10(e)


BLOCK = block_average(7)   # 210 m: one block (⅛ mi = 201 m)
HALF = block_average(3)    # 90 m: half a block, for the inset
log.info('Footprint %d cells; surface median %.1f dB', FOOT.sum(), np.median(surface))

## 5. Sample

In [ ]:
to_albers = Transformer.from_crs('EPSG:4326', CRS, always_xy=True)
to_wgs = Transformer.from_crs(CRS, 'EPSG:4326', always_xy=True)


def sample(lat, lon) -> pd.DataFrame:
    x, y = to_albers.transform(np.asarray(lon, float), np.asarray(lat, float))
    r, c = rowcol(TRANSFORM, x, y)
    r, c = np.asarray(r), np.asarray(c)
    assert ((r >= 0) & (r < SHAPE[0]) & (c >= 0) & (c < SHAPE[1])).all(), 'point outside the clip'
    out = pd.DataFrame({'db': surface[r, c], 'point': combined[r, c], 'dom': dominant[r, c],
                        'block': BLOCK[r, c], 'half': HALF[r, c]})
    for i, m in enumerate(MODES):
        out[m] = by_mode[i, r, c]
    return out


def grid_y_miles(y):
    """Address number north (negative south) to miles north of Madison."""
    return y / grid.NUMBERS_PER_MILE if y >= 0 else (y / grid.FIRST_MILE_SOUTH if y >= -grid.FIRST_MILE_SOUTH else (y + grid.FIRST_MILE_SOUTH) / grid.NUMBERS_PER_MILE - 1)


def from_address(x, y):
    """Inverse of grid.to_address: address units to latitude, longitude."""
    y = np.asarray(y, float)
    north = np.select([y >= 0, y >= -grid.FIRST_MILE_SOUTH], [y / grid.NUMBERS_PER_MILE, y / grid.FIRST_MILE_SOUTH],
                      default=(y + grid.FIRST_MILE_SOUTH) / grid.NUMBERS_PER_MILE - 1)
    east = np.asarray(x, float) / grid.NUMBERS_PER_MILE
    return grid.ORIGIN_LAT + north / grid.MILES_PER_DEG_LAT, grid.ORIGIN_LON + east / grid.MILES_PER_DEG_LON


def in_city(lat, lon) -> np.ndarray:
    x, y = grid.to_address(lat, lon)
    return shapely.contains_xy(city, x, y)


# 'L' stations: the loudest platform of each station
p = platforms[platforms['parent_station'].isin(stations['id'])]
missing = stations[stations['id'].isin(NO_PLATFORM)]
m_lat, m_lon = from_address(missing['x'].values, missing['y'].values)
p = pd.concat([p, pd.DataFrame({'parent_station': missing['id'].values, 'stop_lat': m_lat, 'stop_lon': m_lon})],
              ignore_index=True)
p = pd.concat([p, sample(p['stop_lat'], p['stop_lon'])], axis=1)
loud_platform = p.sort_values('db', ascending=False).drop_duplicates('parent_station')
st = stations.merge(loud_platform[['parent_station', 'db', 'point', 'dom', *MODES]],
                    left_on='id', right_on='parent_station').drop(columns='parent_station')
st['source'] = st['dom'].map(lambda i: MODES[i] if i >= 0 else 'none')

# bus stops in the city
bus = bus[in_city(bus['stop_lat'], bus['stop_lon'])].reset_index(drop=True)
bus = pd.concat([bus, sample(bus['stop_lat'], bus['stop_lon'])], axis=1)

# residents: block internal points
blocks = pd.concat([blocks, sample(blocks['lat'], blocks['lon'])], axis=1)

# Midway monitors: measured aircraft DNL against modelled aviation LAeq,24h
monitors = pd.concat([monitors, sample(monitors['lat'], monitors['lon'])], axis=1)
monitors['measured'] = pd.to_numeric(monitors['avg12'], errors='coerce')
monitors['diff'] = (monitors['aviation'] - monitors['measured']).round(1)
monitors['x'], monitors['y'] = grid.to_address(monitors['lat'], monitors['lon'])
monitors[['monitor', 'community', 'address', 'measured', 'aviation', 'diff']]

## 6. Analyze

In [ ]:
def share(v, w=None, t=LOUD):
    w = np.ones(len(v)) if w is None else np.asarray(w, float)
    return float(w[np.asarray(v) >= t].sum() / w.sum())


P = int(blocks['pop'].sum())
loud_res = blocks[blocks['db'] >= LOUD]
summary = {
    'stations': int(len(st)),
    'stationMedian': round(float(st['db'].median()), 1),
    'stationsLoud': int((st['db'] >= LOUD).sum()),
    'stationsEpa': int((st['db'] >= EPA).sum()),
    'stationsLoudAccessible': int(((st['db'] >= LOUD) & st['ada']).sum()),
    'stationsAccessible': int(st['ada'].sum()),
    'stationsLoudBySource': st[st['db'] >= LOUD]['source'].value_counts().to_dict(),
    'stationsPointLoud': int((st['point'] >= LOUD).sum()),
    'bus': int(len(bus)),
    'busMedian': round(float(bus['db'].median()), 1),
    'busLoud': int((bus['db'] >= LOUD).sum()),
    'busLoudShare': round(share(bus['db']), 4),
    'busEpa': int((bus['db'] >= EPA).sum()),
    'busLoudBySource': bus[bus['db'] >= LOUD]['dom'].map(lambda i: MODES[i]).value_counts().to_dict(),
    'residents': P,
    'residentsBands': {t: round(share(blocks['db'], blocks['pop'], t), 4) for t in CLASSES},
    'residentsLoud': int(loud_res['pop'].sum()),
    'residentsLoudBySource': loud_res.groupby(loud_res['dom'].map(lambda i: MODES[i]))['pop'].sum().astype(int).to_dict(),
    'residentsEpa': int(blocks.loc[blocks['db'] >= EPA, 'pop'].sum()),
    'monitorMeanDiff': round(float(monitors['diff'].mean()), 1),
    'monitorDiffRange': [float(monitors['diff'].min()), float(monitors['diff'].max())],
}
lines = {}
for line in seq:
    ids = {int(i) for seg in seq[line] for i in seg}
    s = st[st['id'].isin(ids)]
    lines[line] = {'stations': int(len(s)), 'loud': int((s['db'] >= LOUD).sum()), 'median': round(float(s['db'].median()), 1)}
summary['lines'] = lines
print(json.dumps(summary, indent=1))
print(st.sort_values('db', ascending=False)[['short', 'lines', 'ada', 'db', 'point', *MODES, 'source']].head(15).round(1).to_string(index=False))
print(st.sort_values('db')[['short', 'lines', 'db']].head(6).round(1).to_string(index=False))

In [ ]:
def words(n: int) -> str:
    return f'{n:,}'


top = st.sort_values('db', ascending=False).iloc[0]
# Stations in expressway medians: the Dan Ryan (Red), the Kennedy and the Eisenhower (Blue)
MEDIAN = {('Red', n) for n in ['Sox-35th', '47th', 'Garfield', '63rd', '69th', '79th', '87th', '95th/Dan Ryan']} | \
         {('Blue', n) for n in ['Addison', 'Irving Park', 'Montrose', 'Jefferson Park', 'Harlem', 'Cumberland',
                                'Clinton', 'UIC-Halsted', 'Racine', 'Illinois Medical District', 'Western', 'Kedzie-Homan',
                                'Pulaski', 'Cicero', 'Austin', 'Oak Park']}
loud = st[st['db'] >= LOUD]
road_loud = loud[loud['source'] == 'road']
assert all((r.lines[0], r.short) in MEDIAN for r in road_loud.itertuples()), 'a road-loud station outside the medians'
summary['stationsLoudMedian'] = int(len(road_loud))
top2 = loud.nlargest(2, 'db')
assert set(top2['lines'].map(tuple)) == {('Orange',)}, top2
FINDING = (f'On an average day, {summary["stationsLoud"]} of Chicago’s {summary["stations"]} ‘L’ stations sit in transport '
           f'noise of {LOUD} dB or more. At {len(road_loud)}, all in expressway medians, the road is the loudest source. '
           f'The loudest of all are on the Orange Line, built along freight railroad lines.')
FINDING

## 7. QA view

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 9))
ax[0].imshow(surface, cmap='Greys', vmin=40, vmax=80)
ax[0].set_title('Combined noise, 90 m median (LAeq,24h)')
ax[1].scatter(monitors['measured'], monitors['aviation'], color='k')
for r in monitors.itertuples():
    ax[1].annotate(r.monitor, (r.measured, r.aviation), fontsize=8)
ax[1].plot([45, 80], [45, 80], color='0.6', lw=0.6)
ax[1].set(xlabel='measured aircraft DNL (CDA, 12 months to Dec 2025)', ylabel='modelled aviation LAeq,24h (BTS 2022)',
          title='Midway monitors: model against measurement')
plt.show()

## 8. Export

`plate.json`: ledger, finding, summary; `stations` (id, short, lines, ada, x, y, dB by mode);
`lattice`: one dot per city block (⅛ mi, address units) inside the city, [x, y, dB, dominant mode
index], dB the energy average of the surface over the block; `inset`: its box and a lattice of two
points per block (1/16 mi), averaged over half a block; `monitors`; `rail` (track,
from Day 08). Keys are camelCase.

In [ ]:
# lattice: one point per block, ⅛ mile, inside the city
xs = np.arange(-14 * LATTICE, 6 * LATTICE + 1) / LATTICE   # the city reaches 4100 E and 13800 S
ys = np.arange(-17 * LATTICE, 12 * LATTICE + 1) / LATTICE
gx, gy = np.meshgrid(xs, ys)
lat = grid.ORIGIN_LAT + gy.ravel() / grid.MILES_PER_DEG_LAT
lon = grid.ORIGIN_LON + gx.ravel() / grid.MILES_PER_DEG_LON
ax_, ay_ = grid.to_address(lat, lon)
keep = shapely.contains_xy(city, ax_, ay_)
lat_s = sample(lat[keep], lon[keep])
lattice = [[round(float(x)), round(float(y)), round(float(d), 1), int(s)]
           for x, y, d, s in zip(ax_[keep], ay_[keep], lat_s['block'], lat_s['dom'])]
log.info('Lattice: %d blocks', len(lattice))
assert keep.sum() > 0 and not shapely.contains_xy(city, ax_[~keep], ay_[~keep]).any()
_lx, _ly = np.array(lattice)[:, 0], np.array(lattice)[:, 1]
log.info('Lattice spans %d to %d E, %d to %d N (address units)', _lx.min(), _lx.max(), _ly.min(), _ly.max())


# inset: the Dan Ryan and the Green Line, 1600 W to 1600 E, 4400 S to 8000 S, two dots per block
INSET = {'west': -1600, 'east': 1600, 'north': -4400, 'south': -8000}
STEP16 = 50  # address units: 1/16 mile
ix = np.arange(INSET['west'] - 400, INSET['east'] + 401, STEP16)
iy_m = np.arange(grid_y_miles(INSET['south'] - 2000), grid_y_miles(INSET['north'] + 2000) + 1e-9, 1 / 16)
gx, gy = np.meshgrid(ix / grid.NUMBERS_PER_MILE, iy_m)
lat = grid.ORIGIN_LAT + gy.ravel() / grid.MILES_PER_DEG_LAT
lon = grid.ORIGIN_LON + gx.ravel() / grid.MILES_PER_DEG_LON
ax_, ay_ = grid.to_address(lat, lon)
keep = shapely.contains_xy(city, ax_, ay_)
ins = sample(lat[keep], lon[keep])
inset_lattice = [[round(float(x)), round(float(y)), round(float(d), 1)] for x, y, d in zip(ax_[keep], ay_[keep], ins['half'])]
log.info('Inset lattice: %d points', len(inset_lattice))

r1 = lambda v: None if pd.isna(v) else round(float(v), 1)
ledger = Ledger(
    source='Bureau of Transportation Statistics, National Transportation Noise Map (2022 release)',
    dataset_id='NTNM 2022',
    url=BTS_PAGE,
    pulled=date(2026, 11, 12),
    units=f'{len(st)} city ‘L’ stations, {len(bus):,} CTA bus stops, {P:,} residents (2020 Census)',
    method=[
        'Modelled road (2023 traffic), rail (2022, incl. the ‘L’ and freight) and aviation (2022) noise, LAeq,24h on 30 m cells, combined by energy',
        f'Each point reads the median within {RADIUS_M} m; a station reads its loudest platform (CTA GTFS {GTFS_DATE:%-d %b %Y})',
        'Map: one dot per block, the energy average of that surface over the block',
        'Midway check: CDA aircraft-noise monitors, 12 months to December 2025',
    ],
    caveats=[
        'A model, not a measurement: buildings and barriers do not block sound in it, so dense streets read louder than they are',
        'Nothing below 45 dB is mapped; local streets without traffic counts are absent',
        'Outdoors only: subway platforms read quiet although tunnels are the loudest ride inside the train',
    ],
)
payload = {
    'sources': {'bts': BTS_PAGE, 'btsDoc': BTS_DOC, 'midway': MDW_REPORT, 'epa': EPA_URL, 'ctaStudy': CTA_STUDY,
                'gtfs': GTFS.name, 'day08': 'days/day08-utopia/output/plate.json'},
    'gtfsDate': GTFS_DATE.isoformat(),
    'thresholds': {'loud': LOUD, 'epa': EPA, 'classes': CLASSES, 'radiusM': RADIUS_M},
    'summary': summary,
    'stations': [
        {'id': str(r.id), 'short': r.short, 'name': r.name, 'lines': r.lines, 'ada': bool(r.ada),
         'x': r1(r.x), 'y': r1(r.y), 'db': r1(r.db), 'point': r1(r.point), 'road': r1(r.road), 'rail': r1(r.rail),
         'aviation': r1(r.aviation), 'source': r.source}
        for r in st.itertuples()
    ],
    'lattice': lattice,
    'inset': {**INSET, 'step': STEP16, 'lattice': inset_lattice},
    'monitors': [
        {'id': r.monitor, 'community': r.community, 'address': r.address, 'x': round(float(r.x)), 'y': round(float(r.y)),
         'measured': r1(r.measured), 'modelled': r1(r.aviation)}
        for r in monitors.itertuples()
    ],
    'rail': rail_track,
}
path = write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload=payload)
assert 'NaN' not in path.read_text(), 'NaN is not valid JSON for the site'
log.info('Wrote %s (%.0f KB)', path.relative_to(DAY_DIR), path.stat().st_size / 1024)